<a href="https://colab.research.google.com/github/alimor37/WAAM_Weld_ML/blob/main/3_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code will do a comparison to see if compact or full chemistry data is better ant of strength using rigid regression model!

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD DATA
# ==================================================

url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(url)

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError("Unexpected Base_Material value")

if df["Weld_Method"].isna().any():
    raise ValueError("Missing Weld_Method value")

# ==================================================
# 2. DEFINE FEATURE SETS
# ==================================================

compact_num = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_num = chemistry + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_num,
    "Full_Chemistry": full_num
}

targets = ["Rp0.2_Mpa", "Rm_Mpa"]

# ==================================================
# 3. DEFINE MODEL PIPELINE
# ==================================================

def make_pipeline(numeric_features):

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, ["Weld_Method"])
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("ridge", Ridge(solver="lsqr"))
    ])

    return model

# ==================================================
# 4. CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

alpha_grid = {
    "ridge__alpha": [
        0.001, 0.01, 0.1,
        1, 10, 100, 1000
    ]
}

prediction_records = []
fold_records = []
repeat_records = []

# ==================================================
# 5. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    valid_df = df.loc[df[target].notna()].copy()
    y = valid_df[target].to_numpy(dtype=float)

    print(f"\n{'=' * 65}")
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print(f"{'=' * 65}")

    # Same outer splits for both feature sets
    outer_splits = list(
        outer_cv.split(valid_df, y)
    )

    for set_name, numeric_features in feature_sets.items():

        selected_features = (
            numeric_features + ["Weld_Method"]
        )

        X = valid_df[selected_features].copy()

        # Predictions from each complete CV repetition
        repetition_predictions = {
            repeat_id: np.full(len(y), np.nan)
            for repeat_id in range(1, 11)
        }

        print(f"\nRunning: {set_name}")

        for fold_index, (train_idx, test_idx) in enumerate(
            outer_splits
        ):

            repeat_id = fold_index // 5 + 1
            fold_id = fold_index % 5 + 1

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y[train_idx]
            y_test = y[test_idx]

            pipeline = make_pipeline(numeric_features)

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=alpha_grid,
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                refit=True,
                n_jobs=1
            )

            # Hyperparameter tuning uses training data only
            search.fit(X_train, y_train)

            # Evaluation uses untouched outer test data
            y_pred = search.predict(X_test)

            best_alpha = search.best_params_[
                "ridge__alpha"
            ]

            repetition_predictions[repeat_id][
                test_idx
            ] = y_pred

            fold_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Train_N": len(train_idx),
                "Test_N": len(test_idx),
                "Best_Alpha": best_alpha,
                "Test_MAE": mean_absolute_error(
                    y_test, y_pred
                )
            })

            for local_idx, actual, predicted in zip(
                test_idx, y_test, y_pred
            ):

                prediction_records.append({
                    "Target": target,
                    "Feature_Set": set_name,
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Excel_Row": int(
                        valid_df.index[local_idx]
                    ) + 2,
                    "Actual": actual,
                    "Predicted": predicted,
                    "Residual": actual - predicted
                })

        # Calculate metrics using all 28
        # out-of-fold predictions in each repetition
        for repeat_id, predictions in (
            repetition_predictions.items()
        ):

            if np.isnan(predictions).any():
                raise RuntimeError(
                    "Incomplete out-of-fold predictions"
                )

            mae = mean_absolute_error(
                y, predictions
            )

            rmse = np.sqrt(
                mean_squared_error(y, predictions)
            )

            r2 = r2_score(y, predictions)

            repeat_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "N": len(y),
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })

        print("Completed successfully.")

# ==================================================
# 6. ORGANIZE RESULTS
# ==================================================

repeat_df = pd.DataFrame(repeat_records)
fold_df = pd.DataFrame(fold_records)
pred_df = pd.DataFrame(prediction_records)

summary = (
    repeat_df
    .groupby(["Target", "Feature_Set"])
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

# ==================================================
# 7. DISPLAY SUMMARY
# ==================================================

print("\n" + "=" * 75)
print("RIDGE NESTED CV — FINAL SUMMARY")
print("=" * 75)

print(
    summary.round(3).to_string(index=False)
)

# ==================================================
# 8. SAVE RESULTS
# ==================================================

output_file = "Ridge_NestedCV_Results.xlsx"

with pd.ExcelWriter(output_file) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    pred_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

print(f"\nSaved: {output_file}")
print("Nested Cross-Validation completed.")
print("Original data unchanged.")



TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running: Compact
Completed successfully.

Running: Full_Chemistry
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28

Running: Compact
Completed successfully.

Running: Full_Chemistry
Completed successfully.

RIDGE NESTED CV — FINAL SUMMARY
   Target    Feature_Set  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa        Compact 28    41.752   4.689     52.673    5.316    0.526  0.099
   Rm_Mpa Full_Chemistry 28    35.014   5.004     46.613    6.123    0.626  0.111
Rp0.2_Mpa        Compact 28    37.595   3.612     53.680    2.697    0.750  0.025
Rp0.2_Mpa Full_Chemistry 28    47.222   6.042     66.777    7.161    0.610  0.084

Saved: Ridge_NestedCV_Results.xlsx
Nested Cross-Validation completed.
Original data unchanged.


Now we do the same but for the PLS model

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cross_decomposition import PLSRegression

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD CLEANED DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL FEATURES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError("Unexpected Base_Material value")

if df["Weld_Method"].isna().any():
    raise ValueError("Missing Weld_Method value")

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. DEFINE PLS PIPELINE
# ==================================================

def make_pls_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    pls_pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "pls",
            PLSRegression(
                scale=False
            )
        )
    ])

    return pls_pipeline

# ==================================================
# 5. CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

component_grid = {
    "pls__n_components": [1, 2, 3, 4, 5, 6]
}

# ==================================================
# 6. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 7. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    # Only exclude rows where this target is unavailable.
    valid_df = df.loc[df[target].notna()].copy()

    y = valid_df[target].to_numpy(dtype=float)

    print("\n" + "=" * 65)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 65)

    # Generate identical outer splits for both feature sets.
    outer_splits = list(
        outer_cv.split(valid_df, y)
    )

    for set_name, numeric_features in feature_sets.items():

        selected_features = (
            numeric_features + ["Weld_Method"]
        )

        X = valid_df[selected_features].copy()

        repetition_predictions = {
            repeat_id: np.full(len(y), np.nan)
            for repeat_id in range(1, 11)
        }

        print(f"\nRunning PLS: {set_name}")

        for fold_index, (train_idx, test_idx) in enumerate(
            outer_splits
        ):

            repeat_id = fold_index // 5 + 1
            fold_id = fold_index % 5 + 1

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y[train_idx]
            y_test = y[test_idx]

            pipeline = make_pls_pipeline(
                numeric_features
            )

            # Find the optimal number of PLS components
            # using only the outer training data.
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=component_grid,
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                refit=True,
                n_jobs=1,
                error_score="raise"
            )

            search.fit(X_train, y_train)

            # Predict the untouched outer test samples.
            y_pred = (
                search.predict(X_test)
                .ravel()
            )

            best_components = search.best_params_[
                "pls__n_components"
            ]

            repetition_predictions[repeat_id][
                test_idx
            ] = y_pred

            fold_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Train_N": len(train_idx),
                "Test_N": len(test_idx),
                "Best_Components": best_components,
                "Test_MAE": mean_absolute_error(
                    y_test, y_pred
                )
            })

            for local_idx, actual, predicted in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target": target,
                    "Feature_Set": set_name,
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Excel_Row": int(
                        valid_df.index[local_idx]
                    ) + 2,
                    "Actual": actual,
                    "Predicted": predicted,
                    "Residual": actual - predicted
                })

        # Evaluate the 28 out-of-fold predictions
        # separately for each complete CV repetition.
        for repeat_id, predictions in (
            repetition_predictions.items()
        ):

            if np.isnan(predictions).any():
                raise RuntimeError(
                    "Incomplete out-of-fold predictions"
                )

            mae = mean_absolute_error(
                y, predictions
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y, predictions
                )
            )

            r2 = r2_score(
                y, predictions
            )

            repeat_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "N": len(y),
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })

        print("Completed successfully.")

# ==================================================
# 8. ORGANIZE RESULTS
# ==================================================

repeat_df = pd.DataFrame(repeat_records)
fold_df = pd.DataFrame(fold_records)
prediction_df = pd.DataFrame(prediction_records)

summary = (
    repeat_df
    .groupby(["Target", "Feature_Set"])
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

# ==================================================
# 9. COMPONENT SELECTION SUMMARY
# ==================================================

component_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Components"
    ])
    .size()
    .reset_index(name="Selection_Count")
)

# ==================================================
# 10. DISPLAY RESULTS
# ==================================================

print("\n" + "=" * 75)
print("PLS NESTED CV — FINAL SUMMARY")
print("=" * 75)

print(
    summary.round(3).to_string(index=False)
)

print("\n" + "=" * 75)
print("PLS COMPONENT SELECTION FREQUENCY")
print("=" * 75)

print(
    component_summary.to_string(index=False)
)

# ==================================================
# 11. SAVE RESULTS
# ==================================================

output_file = "PLS_NestedCV_Results.xlsx"

with pd.ExcelWriter(output_file) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    component_summary.to_excel(
        writer,
        sheet_name="Component_Selection",
        index=False
    )

print(f"\nSaved: {output_file}")
print("PLS Nested Cross-Validation completed.")
print("Original dataset unchanged.")



TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running PLS: Compact
Completed successfully.

Running PLS: Full_Chemistry
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28

Running PLS: Compact
Completed successfully.

Running PLS: Full_Chemistry
Completed successfully.

PLS NESTED CV — FINAL SUMMARY
   Target    Feature_Set  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa        Compact 28    44.946   4.403     57.662    5.578    0.432  0.110
   Rm_Mpa Full_Chemistry 28    35.402   4.489     45.973    5.089    0.638  0.084
Rp0.2_Mpa        Compact 28    39.030   3.800     55.338    3.304    0.734  0.032
Rp0.2_Mpa Full_Chemistry 28    44.320   4.480     61.709    5.301    0.668  0.057

PLS COMPONENT SELECTION FREQUENCY
   Target    Feature_Set  Best_Components  Selection_Count
   Rm_Mpa        Compact                1               12
   Rm_Mpa        Compact                2               17
   Rm_Mpa        Compact                3                1
   Rm_Mpa        Co

In [ ]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD CLEANED DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL FEATURES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError("Unexpected Base_Material value")

if df["Weld_Method"].isna().any():
    raise ValueError("Missing Weld_Method value")

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. DEFINE SVR PIPELINE
# ==================================================

def make_svr_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    svr_pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "svr",
            SVR(kernel="rbf")
        )
    ])

    return svr_pipeline

# ==================================================
# 5. NESTED CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# Small, predefined hyperparameter grid
param_grid = {
    "svr__C": [10, 100, 1000],
    "svr__epsilon": [5, 20],
    "svr__gamma": ["scale", 0.01, 0.1]
}

# ==================================================
# 6. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 7. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    # Exclude rows only when the target is unavailable.
    valid_df = df.loc[df[target].notna()].copy()

    y = valid_df[target].to_numpy(dtype=float)

    print("\n" + "=" * 65)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 65)

    # Same outer splits used in Ridge and PLS
    outer_splits = list(
        outer_cv.split(valid_df, y)
    )

    for set_name, numeric_features in feature_sets.items():

        selected_features = (
            numeric_features + ["Weld_Method"]
        )

        X = valid_df[selected_features].copy()

        repetition_predictions = {
            repeat_id: np.full(len(y), np.nan)
            for repeat_id in range(1, 11)
        }

        print(f"\nRunning SVR: {set_name}")

        for fold_index, (train_idx, test_idx) in enumerate(
            outer_splits
        ):

            repeat_id = fold_index // 5 + 1
            fold_id = fold_index % 5 + 1

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y[train_idx]
            y_test = y[test_idx]

            pipeline = make_svr_pipeline(
                numeric_features
            )

            # Tune parameters using outer training data only.
            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                refit=True,
                n_jobs=1,
                error_score="raise"
            )

            search.fit(X_train, y_train)

            # Evaluate on untouched outer test samples.
            y_pred = search.predict(X_test)

            best_params = search.best_params_

            repetition_predictions[repeat_id][
                test_idx
            ] = y_pred

            fold_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Train_N": len(train_idx),
                "Test_N": len(test_idx),
                "Best_C": best_params["svr__C"],
                "Best_Epsilon": best_params["svr__epsilon"],
                "Best_Gamma": best_params["svr__gamma"],
                "Test_MAE": mean_absolute_error(
                    y_test, y_pred
                )
            })

            for local_idx, actual, predicted in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target": target,
                    "Feature_Set": set_name,
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Excel_Row": int(
                        valid_df.index[local_idx]
                    ) + 2,
                    "Actual": actual,
                    "Predicted": predicted,
                    "Residual": actual - predicted
                })

        # Compute metrics across all out-of-fold
        # predictions for each complete repetition.
        for repeat_id, predictions in (
            repetition_predictions.items()
        ):

            if np.isnan(predictions).any():
                raise RuntimeError(
                    "Incomplete out-of-fold predictions"
                )

            mae = mean_absolute_error(
                y, predictions
            )

            rmse = np.sqrt(
                mean_squared_error(y, predictions)
            )

            r2 = r2_score(
                y, predictions
            )

            repeat_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "N": len(y),
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })

        print("Completed successfully.")

# ==================================================
# 8. ORGANIZE RESULTS
# ==================================================

repeat_df = pd.DataFrame(repeat_records)
fold_df = pd.DataFrame(fold_records)
prediction_df = pd.DataFrame(prediction_records)

summary = (
    repeat_df
    .groupby(["Target", "Feature_Set"])
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

# ==================================================
# 9. HYPERPARAMETER SELECTION FREQUENCY
# ==================================================

parameter_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_C",
        "Best_Epsilon",
        "Best_Gamma"
    ])
    .size()
    .reset_index(name="Selection_Count")
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[True, True, False]
    )
)

# ==================================================
# 10. DISPLAY FINAL SUMMARY
# ==================================================

print("\n" + "=" * 75)
print("SVR NESTED CV — FINAL SUMMARY")
print("=" * 75)

print(
    summary.round(3).to_string(index=False)
)

print("\n" + "=" * 75)
print("SVR HYPERPARAMETER SELECTION — TOP COMBINATIONS")
print("=" * 75)

print(
    parameter_summary
    .groupby(["Target", "Feature_Set"])
    .head(3)
    .to_string(index=False)
)

# ==================================================
# 11. SAVE RESULTS
# ==================================================

output_file = "SVR_NestedCV_Results.xlsx"

with pd.ExcelWriter(output_file) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    parameter_summary.to_excel(
        writer,
        sheet_name="Parameter_Selection",
        index=False
    )

print(f"\nSaved: {output_file}")
print("SVR Nested Cross-Validation completed.")
print("Original dataset unchanged.")



TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running SVR: Compact
Completed successfully.

Running SVR: Full_Chemistry
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28

Running SVR: Compact
Completed successfully.

Running SVR: Full_Chemistry
Completed successfully.

SVR NESTED CV — FINAL SUMMARY
   Target    Feature_Set  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa        Compact 28    41.218   3.917     54.864    4.412    0.487  0.083
   Rm_Mpa Full_Chemistry 28    37.807   3.107     50.361    3.476    0.569  0.059
Rp0.2_Mpa        Compact 28    39.123   5.663     59.645    6.021    0.689  0.064
Rp0.2_Mpa Full_Chemistry 28    42.638   3.220     60.534    3.813    0.681  0.040

SVR HYPERPARAMETER SELECTION — TOP COMBINATIONS
   Target    Feature_Set  Best_C  Best_Epsilon Best_Gamma  Selection_Count
   Rm_Mpa        Compact    1000             5       0.01               15
   Rm_Mpa        Compact    1000            20        0.1               14
   Rm_Mpa       

Now we will test the Dummy mode (The most basic model to only compare the base model with the more advanced model that we had before like SVR or ridge)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 2. SAME OUTER CV AS PREVIOUS MODELS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

repeat_records = []
prediction_records = []
fold_records = []

# ==================================================
# 3. RUN DUMMY BASELINE
# ==================================================

for target in targets:

    valid_df = df.loc[df[target].notna()].copy()

    y = valid_df[target].to_numpy(dtype=float)

    # DummyRegressor does not use real features,
    # so a single constant input column is enough.
    X = np.zeros((len(y), 1))

    print("\n" + "=" * 65)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(y)}")
    print("=" * 65)

    outer_splits = list(
        outer_cv.split(X, y)
    )

    repetition_predictions = {
        repeat_id: np.full(len(y), np.nan)
        for repeat_id in range(1, 11)
    }

    for fold_index, (train_idx, test_idx) in enumerate(
        outer_splits
    ):

        repeat_id = fold_index // 5 + 1
        fold_id = fold_index % 5 + 1

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # Baseline model:
        # always predicts mean of training target values
        model = DummyRegressor(
            strategy="mean"
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        repetition_predictions[
            repeat_id
        ][test_idx] = y_pred

        fold_records.append({
            "Target": target,
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Train_N": len(train_idx),
            "Test_N": len(test_idx),
            "Training_Mean": float(model.constant_.ravel()[0]),
            "Test_MAE": mean_absolute_error(
                y_test,
                y_pred
            )
        })

        for local_idx, actual, predicted in zip(
            test_idx,
            y_test,
            y_pred
        ):

            prediction_records.append({
                "Target": target,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Excel_Row": int(
                    valid_df.index[local_idx]
                ) + 2,
                "Actual": actual,
                "Predicted": predicted,
                "Residual": actual - predicted
            })

    # ==================================================
    # 4. METRICS PER FULL REPETITION
    # ==================================================

    for repeat_id, predictions in (
        repetition_predictions.items()
    ):

        if np.isnan(predictions).any():
            raise RuntimeError(
                "Incomplete predictions found"
            )

        mae = mean_absolute_error(
            y,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y,
                predictions
            )
        )

        r2 = r2_score(
            y,
            predictions
        )

        repeat_records.append({
            "Target": target,
            "Repeat": repeat_id,
            "N": len(y),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    print("Completed successfully.")

# ==================================================
# 5. CREATE DATAFRAMES
# ==================================================

repeat_df = pd.DataFrame(
    repeat_records
)

fold_df = pd.DataFrame(
    fold_records
)

prediction_df = pd.DataFrame(
    prediction_records
)

# ==================================================
# 6. FINAL SUMMARY
# ==================================================

summary = (
    repeat_df
    .groupby("Target")
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

print("\n" + "=" * 75)
print("DUMMY BASELINE — FINAL SUMMARY")
print("=" * 75)

print(
    summary.round(3).to_string(index=False)
)

# ==================================================
# 7. SAVE RESULTS
# ==================================================

output_file = "Dummy_Baseline_Results.xlsx"

with pd.ExcelWriter(
    output_file
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

print(f"\nSaved: {output_file}")
print("Dummy baseline completed.")
print("Original dataset unchanged.")


TARGET: Rp0.2_Mpa
VALID SAMPLES: 28
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28
Completed successfully.

DUMMY BASELINE — FINAL SUMMARY
   Target  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa 28    68.221   1.762     80.018    2.037   -0.085  0.056
Rp0.2_Mpa 28    92.238   2.136    112.113    2.384   -0.089  0.046

Saved: Dummy_Baseline_Results.xlsx
Dummy baseline completed.
Original dataset unchanged.


Now lets tune Ridge hyperparameters:

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD CLEANED DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL VARIABLES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError(
        "Unexpected Base_Material value"
    )

if df["Weld_Method"].isna().any():
    raise ValueError(
        "Missing Weld_Method value"
    )

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. CREATE RIDGE PIPELINE
# ==================================================

def make_ridge_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    ridge_pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "ridge",
            Ridge(
                max_iter=100000,
                tol=1e-6,
                random_state=42
            )
        )
    ])

    return ridge_pipeline

# ==================================================
# 5. NESTED CV SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# ==================================================
# 6. GRID SEARCH PARAMETERS
# ==================================================

param_grid = {

    "ridge__alpha": [
        0.0001,
        0.001,
        0.01,
        0.1,
        0.3,
        1,
        3,
        10,
        30,
        100,
        300,
        1000,
        3000,
        10000
    ],

    "ridge__solver": [
        "auto",
        "svd",
        "cholesky",
        "lsqr",
        "sag",
        "saga"
    ]
}

# ==================================================
# 7. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 8. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    valid_df = df.loc[
        df[target].notna()
    ].copy()

    y = valid_df[
        target
    ].to_numpy(dtype=float)

    print("\n" + "=" * 70)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 70)

    # Same outer CV splits as previous models
    outer_splits = list(
        outer_cv.split(valid_df, y)
    )

    for set_name, numeric_features in (
        feature_sets.items()
    ):

        selected_features = (
            numeric_features
            + ["Weld_Method"]
        )

        X = valid_df[
            selected_features
        ].copy()

        repetition_predictions = {
            repeat_id: np.full(
                len(y),
                np.nan
            )
            for repeat_id in range(1, 11)
        }

        print(
            f"\nRunning Tuned Ridge: "
            f"{set_name}"
        )

        for fold_index, (
            train_idx,
            test_idx
        ) in enumerate(outer_splits):

            repeat_id = (
                fold_index // 5 + 1
            )

            fold_id = (
                fold_index % 5 + 1
            )

            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_train = y[
                train_idx
            ]

            y_test = y[
                test_idx
            ]

            pipeline = (
                make_ridge_pipeline(
                    numeric_features
                )
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring=(
                    "neg_mean_absolute_error"
                ),
                cv=inner_cv,
                refit=True,
                n_jobs=-1,
                error_score="raise"
            )

            # Hyperparameter tuning:
            # outer training samples only
            search.fit(
                X_train,
                y_train
            )

            # Untouched outer test samples
            y_pred = search.predict(
                X_test
            )

            best_alpha = (
                search.best_params_[
                    "ridge__alpha"
                ]
            )

            best_solver = (
                search.best_params_[
                    "ridge__solver"
                ]
            )

            repetition_predictions[
                repeat_id
            ][test_idx] = y_pred

            # --------------------------------
            # Save fold information
            # --------------------------------

            fold_records.append({
                "Target": target,
                "Feature_Set": set_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Train_N": len(train_idx),
                "Test_N": len(test_idx),
                "Best_Alpha": best_alpha,
                "Best_Solver": best_solver,
                "Test_MAE":
                    mean_absolute_error(
                        y_test,
                        y_pred
                    )
            })

            # --------------------------------
            # Save predictions
            # --------------------------------

            for (
                local_idx,
                actual,
                predicted
            ) in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target":
                        target,
                    "Feature_Set":
                        set_name,
                    "Repeat":
                        repeat_id,
                    "Fold":
                        fold_id,
                    "Excel_Row":
                        int(
                            valid_df.index[
                                local_idx
                            ]
                        ) + 2,
                    "Actual":
                        actual,
                    "Predicted":
                        predicted,
                    "Residual":
                        actual - predicted
                })

        # ==================================================
        # 9. METRICS FOR EACH COMPLETE REPETITION
        # ==================================================

        for (
            repeat_id,
            predictions
        ) in repetition_predictions.items():

            if np.isnan(
                predictions
            ).any():

                raise RuntimeError(
                    "Incomplete OOF predictions"
                )

            mae = (
                mean_absolute_error(
                    y,
                    predictions
                )
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y,
                    predictions
                )
            )

            r2 = r2_score(
                y,
                predictions
            )

            repeat_records.append({
                "Target":
                    target,
                "Feature_Set":
                    set_name,
                "Repeat":
                    repeat_id,
                "N":
                    len(y),
                "MAE":
                    mae,
                "RMSE":
                    rmse,
                "R2":
                    r2
            })

        print(
            "Completed successfully."
        )

# ==================================================
# 10. ORGANIZE RESULTS
# ==================================================

repeat_df = pd.DataFrame(
    repeat_records
)

fold_df = pd.DataFrame(
    fold_records
)

prediction_df = pd.DataFrame(
    prediction_records
)

# ==================================================
# 11. FINAL PERFORMANCE SUMMARY
# ==================================================

summary = (
    repeat_df
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

# ==================================================
# 12. ALPHA SELECTION FREQUENCY
# ==================================================

alpha_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Alpha"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 13. SOLVER SELECTION FREQUENCY
# ==================================================

solver_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Solver"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 14. ALPHA + SOLVER COMBINATION
# ==================================================

combination_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Alpha",
        "Best_Solver"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 15. DISPLAY RESULTS
# ==================================================

print("\n" + "=" * 80)
print(
    "TUNED RIDGE NESTED CV — FINAL SUMMARY"
)
print("=" * 80)

print(
    summary
    .round(3)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print(
    "MOST FREQUENT ALPHA VALUES"
)
print("=" * 80)

print(
    alpha_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(index=False)
)

print("\n" + "=" * 80)
print(
    "SOLVER SELECTION FREQUENCY"
)
print("=" * 80)

print(
    solver_summary
    .to_string(index=False)
)

print("\n" + "=" * 80)
print(
    "TOP ALPHA + SOLVER COMBINATIONS"
)
print("=" * 80)

print(
    combination_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(index=False)
)

# ==================================================
# 16. SAVE EVERYTHING
# ==================================================

output_file = (
    "Ridge_Tuned_Alpha_Solver_"
    "NestedCV_Results.xlsx"
)

with pd.ExcelWriter(
    output_file
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    alpha_summary.to_excel(
        writer,
        sheet_name="Alpha_Selection",
        index=False
    )

    solver_summary.to_excel(
        writer,
        sheet_name="Solver_Selection",
        index=False
    )

    combination_summary.to_excel(
        writer,
        sheet_name="Alpha_Solver_Selection",
        index=False
    )

print(
    f"\nSaved: {output_file}"
)

print(
    "Tuned Ridge Nested CV completed."
)

print(
    "Original dataset unchanged."
)


TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running Tuned Ridge: Compact
Completed successfully.

Running Tuned Ridge: Full_Chemistry
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28

Running Tuned Ridge: Compact
Completed successfully.

Running Tuned Ridge: Full_Chemistry
Completed successfully.

TUNED RIDGE NESTED CV — FINAL SUMMARY
   Target    Feature_Set  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa        Compact 28    41.093   3.972     51.995    4.861    0.539  0.086
   Rm_Mpa Full_Chemistry 28    35.459   5.016     46.994    5.527    0.621  0.099
Rp0.2_Mpa        Compact 28    37.856   3.818     54.078    2.957    0.746  0.028
Rp0.2_Mpa Full_Chemistry 28    45.659   6.097     64.732    7.232    0.633  0.083

MOST FREQUENT ALPHA VALUES
   Target    Feature_Set  Best_Alpha  Selection_Count
   Rm_Mpa        Compact        3.00               17
   Rm_Mpa        Compact       10.00                9
   Rm_Mpa        Compact        0.30                7
   Rm_

Now lets tune the hyperparameter of PLS

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cross_decomposition import PLSRegression

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 1. LOAD DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL VARIABLES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError("Unexpected Base_Material value")

if df["Weld_Method"].isna().any():
    raise ValueError("Missing Weld_Method value")

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. CREATE PLS PIPELINE
# ==================================================

def make_pls_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    pls_pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "pls",
            PLSRegression()
        )
    ])

    return pls_pipeline

# ==================================================
# 5. CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# ==================================================
# 6. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 7. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    valid_df = df.loc[
        df[target].notna()
    ].copy()

    y = valid_df[
        target
    ].to_numpy(dtype=float)

    print("\n" + "=" * 70)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 70)

    outer_splits = list(
        outer_cv.split(valid_df, y)
    )

    for set_name, numeric_features in feature_sets.items():

        selected_features = (
            numeric_features
            + ["Weld_Method"]
        )

        X = valid_df[
            selected_features
        ].copy()

        # ------------------------------------------
        # Different component ranges by feature set
        # ------------------------------------------

        if set_name == "Compact":

            component_values = [
                1, 2, 3, 4, 5, 6
            ]

        else:

            component_values = [
                1, 2, 3, 4, 5,
                6, 7, 8, 9, 10
            ]

        # ------------------------------------------
        # Hyperparameter grid
        # ------------------------------------------

        param_grid = {

            "pls__n_components":
                component_values,

            "pls__scale": [
                False,
                True
            ],

            "pls__max_iter": [
                300,
                1000
            ],

            "pls__tol": [
                1e-4,
                1e-6
            ]
        }

        repetition_predictions = {
            repeat_id: np.full(
                len(y),
                np.nan
            )
            for repeat_id in range(1, 11)
        }

        print(
            f"\nRunning Tuned PLS: "
            f"{set_name}"
        )

        for fold_index, (
            train_idx,
            test_idx
        ) in enumerate(outer_splits):

            repeat_id = (
                fold_index // 5 + 1
            )

            fold_id = (
                fold_index % 5 + 1
            )

            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_train = y[
                train_idx
            ]

            y_test = y[
                test_idx
            ]

            pipeline = (
                make_pls_pipeline(
                    numeric_features
                )
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                refit=True,
                n_jobs=-1,
                error_score="raise"
            )

            # Inner-loop tuning:
            # only outer-training data are used
            search.fit(
                X_train,
                y_train
            )

            # Evaluation on untouched outer-test data
            y_pred = (
                search.predict(
                    X_test
                )
                .ravel()
            )

            best_params = (
                search.best_params_
            )

            best_components = (
                best_params[
                    "pls__n_components"
                ]
            )

            best_scale = (
                best_params[
                    "pls__scale"
                ]
            )

            best_max_iter = (
                best_params[
                    "pls__max_iter"
                ]
            )

            best_tol = (
                best_params[
                    "pls__tol"
                ]
            )

            repetition_predictions[
                repeat_id
            ][test_idx] = y_pred

            # --------------------------------------
            # Save fold-level information
            # --------------------------------------

            fold_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "Fold":
                    fold_id,

                "Train_N":
                    len(train_idx),

                "Test_N":
                    len(test_idx),

                "Best_Components":
                    best_components,

                "Best_Scale":
                    best_scale,

                "Best_Max_Iter":
                    best_max_iter,

                "Best_Tol":
                    best_tol,

                "Test_MAE":
                    mean_absolute_error(
                        y_test,
                        y_pred
                    )
            })

            # --------------------------------------
            # Save prediction-level information
            # --------------------------------------

            for (
                local_idx,
                actual,
                predicted
            ) in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target":
                        target,

                    "Feature_Set":
                        set_name,

                    "Repeat":
                        repeat_id,

                    "Fold":
                        fold_id,

                    "Excel_Row":
                        int(
                            valid_df.index[
                                local_idx
                            ]
                        ) + 2,

                    "Actual":
                        actual,

                    "Predicted":
                        predicted,

                    "Residual":
                        actual - predicted
                })

        # ==================================================
        # 8. METRICS FOR EACH COMPLETE CV REPETITION
        # ==================================================

        for (
            repeat_id,
            predictions
        ) in repetition_predictions.items():

            if np.isnan(
                predictions
            ).any():

                raise RuntimeError(
                    "Incomplete OOF predictions"
                )

            mae = (
                mean_absolute_error(
                    y,
                    predictions
                )
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y,
                    predictions
                )
            )

            r2 = (
                r2_score(
                    y,
                    predictions
                )
            )

            repeat_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "N":
                    len(y),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            })

        print("Completed successfully.")

# ==================================================
# 9. CREATE RESULT DATAFRAMES
# ==================================================

repeat_df = pd.DataFrame(
    repeat_records
)

fold_df = pd.DataFrame(
    fold_records
)

prediction_df = pd.DataFrame(
    prediction_records
)

# ==================================================
# 10. FINAL PERFORMANCE SUMMARY
# ==================================================

summary = (
    repeat_df
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .agg(
        N=("N", "first"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std")
    )
    .reset_index()
)

# ==================================================
# 11. COMPONENT SELECTION FREQUENCY
# ==================================================

component_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Components"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 12. SCALE SELECTION FREQUENCY
# ==================================================

scale_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Scale"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 13. MAX_ITER SELECTION FREQUENCY
# ==================================================

max_iter_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Max_Iter"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 14. TOL SELECTION FREQUENCY
# ==================================================

tol_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Tol"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 15. FULL PARAMETER COMBINATION SUMMARY
# ==================================================

parameter_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Components",
        "Best_Scale",
        "Best_Max_Iter",
        "Best_Tol"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 16. DISPLAY RESULTS
# ==================================================

print("\n" + "=" * 85)
print(
    "TUNED PLS NESTED CV — FINAL SUMMARY"
)
print("=" * 85)

print(
    summary
    .round(3)
    .to_string(index=False)
)

print("\n" + "=" * 85)
print(
    "MOST FREQUENT COMPONENT COUNTS"
)
print("=" * 85)

print(
    component_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(index=False)
)

print("\n" + "=" * 85)
print(
    "SCALE SELECTION FREQUENCY"
)
print("=" * 85)

print(
    scale_summary
    .to_string(index=False)
)

print("\n" + "=" * 85)
print(
    "MAX_ITER SELECTION FREQUENCY"
)
print("=" * 85)

print(
    max_iter_summary
    .to_string(index=False)
)

print("\n" + "=" * 85)
print(
    "TOL SELECTION FREQUENCY"
)
print("=" * 85)

print(
    tol_summary
    .to_string(index=False)
)

print("\n" + "=" * 100)
print(
    "TOP PLS HYPERPARAMETER COMBINATIONS"
)
print("=" * 100)

print(
    parameter_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(10)
    .to_string(index=False)
)

# ==================================================
# 17. SAVE RESULTS
# ==================================================

output_file = (
    "PLS_Tuned_FullGrid_NestedCV_Results.xlsx"
)

with pd.ExcelWriter(
    output_file
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    component_summary.to_excel(
        writer,
        sheet_name="Component_Selection",
        index=False
    )

    scale_summary.to_excel(
        writer,
        sheet_name="Scale_Selection",
        index=False
    )

    max_iter_summary.to_excel(
        writer,
        sheet_name="MaxIter_Selection",
        index=False
    )

    tol_summary.to_excel(
        writer,
        sheet_name="Tol_Selection",
        index=False
    )

    parameter_summary.to_excel(
        writer,
        sheet_name="Parameter_Selection",
        index=False
    )

print(
    f"\nSaved: {output_file}"
)

print(
    "Full-grid PLS Nested CV completed."
)

print(
    "Original dataset unchanged."
)


TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running Tuned PLS: Compact
Completed successfully.

Running Tuned PLS: Full_Chemistry
Completed successfully.

TARGET: Rm_Mpa
VALID SAMPLES: 28

Running Tuned PLS: Compact
Completed successfully.

Running Tuned PLS: Full_Chemistry
Completed successfully.

TUNED PLS NESTED CV — FINAL SUMMARY
   Target    Feature_Set  N  MAE_Mean  MAE_SD  RMSE_Mean  RMSE_SD  R2_Mean  R2_SD
   Rm_Mpa        Compact 28    44.456   2.614     56.573    4.311    0.455  0.084
   Rm_Mpa Full_Chemistry 28    33.212   3.056     43.475    3.347    0.678  0.050
Rp0.2_Mpa        Compact 28    35.621   3.206     53.145    2.985    0.755  0.028
Rp0.2_Mpa Full_Chemistry 28    38.590   4.476     55.260    7.483    0.731  0.078

MOST FREQUENT COMPONENT COUNTS
   Target    Feature_Set  Best_Components  Selection_Count
   Rm_Mpa        Compact                2               21
   Rm_Mpa        Compact                1                8
   Rm_Mpa        Compact                3          

Lets tune the SVR

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 0. START TIMER
# ==================================================

start_time = time.time()

# ==================================================
# 1. LOAD CLEANED DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL VARIABLES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError(
        "Unexpected Base_Material value"
    )

if df["Weld_Method"].isna().any():
    raise ValueError(
        "Missing Weld_Method value"
    )

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. CREATE RBF-SVR PIPELINE
# ==================================================

def make_svr_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    svr_pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "svr",
            SVR(
                kernel="rbf"
            )
        )
    ])

    return svr_pipeline

# ==================================================
# 5. CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# ==================================================
# 6. HYPERPARAMETER GRID
# ==================================================

param_grid = {

    "svr__C": [
        100,
        300,
        1000,
        3000
    ],

    "svr__epsilon": [
        5,
        10,
        20
    ],

    "svr__gamma": [
        "scale",
        0.01,
        0.03,
        0.1
    ]
}

# 4 × 3 × 4 = 48 combinations

# ==================================================
# 7. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 8. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    valid_df = df.loc[
        df[target].notna()
    ].copy()

    y = valid_df[
        target
    ].to_numpy(
        dtype=float
    )

    print("\n" + "=" * 70)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 70)

    # Same outer split structure
    # as our previous models
    outer_splits = list(
        outer_cv.split(
            valid_df,
            y
        )
    )

    for (
        set_name,
        numeric_features
    ) in feature_sets.items():

        selected_features = (
            numeric_features
            + ["Weld_Method"]
        )

        X = valid_df[
            selected_features
        ].copy()

        repetition_predictions = {
            repeat_id:
                np.full(
                    len(y),
                    np.nan
                )

            for repeat_id
            in range(1, 11)
        }

        print(
            f"\nRunning Tuned RBF-SVR: "
            f"{set_name}"
        )

        # ==================================================
        # OUTER CV
        # ==================================================

        for fold_index, (
            train_idx,
            test_idx
        ) in enumerate(
            outer_splits
        ):

            repeat_id = (
                fold_index // 5 + 1
            )

            fold_id = (
                fold_index % 5 + 1
            )

            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_train = y[
                train_idx
            ]

            y_test = y[
                test_idx
            ]

            pipeline = (
                make_svr_pipeline(
                    numeric_features
                )
            )

            # ----------------------------------------------
            # INNER GRID SEARCH
            # ----------------------------------------------

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,

                scoring=(
                    "neg_mean_absolute_error"
                ),

                cv=inner_cv,

                refit=True,

                n_jobs=-1,

                error_score="raise",

                verbose=0
            )

            # Hyperparameter tuning uses
            # outer-training data only
            search.fit(
                X_train,
                y_train
            )

            # ----------------------------------------------
            # OUTER TEST PREDICTION
            # ----------------------------------------------

            y_pred = (
                search.predict(
                    X_test
                )
            )

            best_params = (
                search.best_params_
            )

            best_C = (
                best_params[
                    "svr__C"
                ]
            )

            best_epsilon = (
                best_params[
                    "svr__epsilon"
                ]
            )

            best_gamma = (
                best_params[
                    "svr__gamma"
                ]
            )

            repetition_predictions[
                repeat_id
            ][test_idx] = y_pred

            # ----------------------------------------------
            # SAVE FOLD DETAILS
            # ----------------------------------------------

            fold_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "Fold":
                    fold_id,

                "Train_N":
                    len(train_idx),

                "Test_N":
                    len(test_idx),

                "Best_C":
                    best_C,

                "Best_Epsilon":
                    best_epsilon,

                "Best_Gamma":
                    best_gamma,

                "Test_MAE":
                    mean_absolute_error(
                        y_test,
                        y_pred
                    )
            })

            # ----------------------------------------------
            # SAVE INDIVIDUAL PREDICTIONS
            # ----------------------------------------------

            for (
                local_idx,
                actual,
                predicted
            ) in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target":
                        target,

                    "Feature_Set":
                        set_name,

                    "Repeat":
                        repeat_id,

                    "Fold":
                        fold_id,

                    "Excel_Row":
                        int(
                            valid_df.index[
                                local_idx
                            ]
                        ) + 2,

                    "Actual":
                        actual,

                    "Predicted":
                        predicted,

                    "Residual":
                        actual - predicted
                })

            # ----------------------------------------------
            # SHOW PROGRESS
            # ----------------------------------------------

            if fold_id == 5:

                elapsed_min = (
                    time.time()
                    - start_time
                ) / 60

                print(
                    f"  Repeat "
                    f"{repeat_id}/10 completed "
                    f"| Elapsed: "
                    f"{elapsed_min:.1f} min"
                )

        # ==================================================
        # 9. METRICS PER COMPLETE REPETITION
        # ==================================================

        for (
            repeat_id,
            predictions
        ) in repetition_predictions.items():

            if np.isnan(
                predictions
            ).any():

                raise RuntimeError(
                    "Incomplete OOF predictions"
                )

            mae = (
                mean_absolute_error(
                    y,
                    predictions
                )
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y,
                    predictions
                )
            )

            r2 = (
                r2_score(
                    y,
                    predictions
                )
            )

            repeat_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "N":
                    len(y),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            })

        print(
            "Completed successfully."
        )

# ==================================================
# 10. CREATE RESULT DATAFRAMES
# ==================================================

repeat_df = pd.DataFrame(
    repeat_records
)

fold_df = pd.DataFrame(
    fold_records
)

prediction_df = pd.DataFrame(
    prediction_records
)

# ==================================================
# 11. FINAL PERFORMANCE SUMMARY
# ==================================================

summary = (
    repeat_df
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .agg(
        N=("N", "first"),

        MAE_Mean=(
            "MAE",
            "mean"
        ),

        MAE_SD=(
            "MAE",
            "std"
        ),

        RMSE_Mean=(
            "RMSE",
            "mean"
        ),

        RMSE_SD=(
            "RMSE",
            "std"
        ),

        R2_Mean=(
            "R2",
            "mean"
        ),

        R2_SD=(
            "R2",
            "std"
        )
    )
    .reset_index()
)

# ==================================================
# 12. C SELECTION FREQUENCY
# ==================================================

C_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_C"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 13. EPSILON SELECTION FREQUENCY
# ==================================================

epsilon_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Epsilon"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 14. GAMMA SELECTION FREQUENCY
# ==================================================

gamma_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_Gamma"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 15. FULL PARAMETER COMBINATIONS
# ==================================================

parameter_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_C",
        "Best_Epsilon",
        "Best_Gamma"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 16. DISPLAY RESULTS
# ==================================================

print("\n" + "=" * 90)

print(
    "TUNED RBF-SVR NESTED CV — FINAL SUMMARY"
)

print("=" * 90)

print(
    summary
    .round(3)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 90)

print(
    "MOST FREQUENT C VALUES"
)

print("=" * 90)

print(
    C_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 90)

print(
    "MOST FREQUENT EPSILON VALUES"
)

print("=" * 90)

print(
    epsilon_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 90)

print(
    "MOST FREQUENT GAMMA VALUES"
)

print("=" * 90)

print(
    gamma_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(5)
    .to_string(
        index=False
    )
)

print("\n" + "=" * 100)

print(
    "TOP RBF-SVR HYPERPARAMETER COMBINATIONS"
)

print("=" * 100)

print(
    parameter_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(10)
    .to_string(
        index=False
    )
)

# ==================================================
# 17. SAVE RESULTS
# ==================================================

output_file = (
    "SVR_Tuned_RBF_"
    "NestedCV_Results.xlsx"
)

with pd.ExcelWriter(
    output_file
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    C_summary.to_excel(
        writer,
        sheet_name="C_Selection",
        index=False
    )

    epsilon_summary.to_excel(
        writer,
        sheet_name="Epsilon_Selection",
        index=False
    )

    gamma_summary.to_excel(
        writer,
        sheet_name="Gamma_Selection",
        index=False
    )

    parameter_summary.to_excel(
        writer,
        sheet_name="Parameter_Selection",
        index=False
    )

# ==================================================
# 18. TOTAL RUNNING TIME
# ==================================================

total_minutes = (
    time.time()
    - start_time
) / 60

print(
    f"\nSaved: {output_file}"
)

print(
    "Tuned RBF-SVR Nested CV completed."
)

print(
    "Original dataset unchanged."
)

print(
    f"Total running time: "
    f"{total_minutes:.1f} minutes"
)


TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running Tuned RBF-SVR: Compact
  Repeat 1/10 completed | Elapsed: 0.3 min
  Repeat 2/10 completed | Elapsed: 0.5 min
  Repeat 3/10 completed | Elapsed: 0.7 min
  Repeat 4/10 completed | Elapsed: 0.9 min
  Repeat 5/10 completed | Elapsed: 1.2 min
  Repeat 6/10 completed | Elapsed: 1.4 min
  Repeat 7/10 completed | Elapsed: 1.6 min
  Repeat 8/10 completed | Elapsed: 1.9 min
  Repeat 9/10 completed | Elapsed: 2.1 min
  Repeat 10/10 completed | Elapsed: 2.3 min
Completed successfully.

Running Tuned RBF-SVR: Full_Chemistry
  Repeat 1/10 completed | Elapsed: 2.6 min
  Repeat 2/10 completed | Elapsed: 2.8 min
  Repeat 3/10 completed | Elapsed: 3.0 min
  Repeat 4/10 completed | Elapsed: 3.2 min
  Repeat 5/10 completed | Elapsed: 3.4 min
  Repeat 6/10 completed | Elapsed: 3.7 min
  Repeat 7/10 completed | Elapsed: 3.9 min
  Repeat 8/10 completed | Elapsed: 4.1 min
  Repeat 9/10 completed | Elapsed: 4.3 min
  Repeat 10/10 completed | Elapsed: 4.6 min
Comple

This is a hyperparamet tuning for SVR but with poly as for the kernel

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==================================================
# 0. START TIMER
# ==================================================

start_time = time.time()

# ==================================================
# 1. LOAD CLEANED DATA
# ==================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df = pd.read_excel(github_url)

# ==================================================
# 2. PREPARE CATEGORICAL VARIABLES
# ==================================================

base_mapping = {
    "Weldox 700": 0,
    "Weldox 1100": 1
}

df["Base1100"] = (
    df["Base_Material"]
    .astype("string")
    .str.strip()
    .map(base_mapping)
)

df["Weld_Method"] = (
    df["Weld_Method"]
    .astype("string")
    .str.strip()
)

if df["Base1100"].isna().any():
    raise ValueError(
        "Unexpected Base_Material value"
    )

if df["Weld_Method"].isna().any():
    raise ValueError(
        "Missing Weld_Method value"
    )

# ==================================================
# 3. DEFINE FEATURE SETS
# ==================================================

compact_numeric = [
    "Pcm",
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

chemistry_features = [
    "C", "Si", "Mn", "P", "S",
    "Cr", "Ni", "Mo", "V", "Nb",
    "Cu", "Al", "Ti", "B", "O", "N"
]

full_numeric = chemistry_features + [
    "t85_s",
    "No_of_Beads",
    "Dilution_Ni_pct",
    "Base1100"
]

feature_sets = {
    "Compact": compact_numeric,
    "Full_Chemistry": full_numeric
}

targets = [
    "Rp0.2_Mpa",
    "Rm_Mpa"
]

# ==================================================
# 4. CREATE POLYNOMIAL-SVR PIPELINE
# ==================================================

def make_poly_svr_pipeline(numeric_features):

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    categorical_pipeline = Pipeline([
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                ["Weld_Method"]
            )
        ],
        sparse_threshold=0
    )

    model = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "svr",
            SVR(
                kernel="poly"
            )
        )
    ])

    return model

# ==================================================
# 5. CROSS-VALIDATION SETTINGS
# ==================================================

outer_cv = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

inner_cv = KFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# ==================================================
# 6. POLYNOMIAL SVR GRID
# ==================================================

param_grid = {

    "svr__C": [
        100,
        300,
        1000,
        3000
    ],

    "svr__epsilon": [
        5,
        10,
        20
    ],

    "svr__gamma": [
        "scale",
        0.01,
        0.03,
        0.1
    ],

    "svr__degree": [
        2,
        3
    ],

    "svr__coef0": [
        0,
        1
    ]
}

# 4 × 3 × 4 × 2 × 2 = 192 combinations

# ==================================================
# 7. RESULT CONTAINERS
# ==================================================

fold_records = []
prediction_records = []
repeat_records = []

# ==================================================
# 8. NESTED CROSS-VALIDATION
# ==================================================

for target in targets:

    valid_df = df.loc[
        df[target].notna()
    ].copy()

    y = valid_df[
        target
    ].to_numpy(dtype=float)

    print("\n" + "=" * 70)
    print(f"TARGET: {target}")
    print(f"VALID SAMPLES: {len(valid_df)}")
    print("=" * 70)

    outer_splits = list(
        outer_cv.split(
            valid_df,
            y
        )
    )

    for (
        set_name,
        numeric_features
    ) in feature_sets.items():

        selected_features = (
            numeric_features
            + ["Weld_Method"]
        )

        X = valid_df[
            selected_features
        ].copy()

        repetition_predictions = {
            repeat_id:
                np.full(
                    len(y),
                    np.nan
                )
            for repeat_id
            in range(1, 11)
        }

        print(
            f"\nRunning Tuned Poly-SVR: "
            f"{set_name}"
        )

        # ==================================================
        # OUTER CV
        # ==================================================

        for fold_index, (
            train_idx,
            test_idx
        ) in enumerate(
            outer_splits
        ):

            repeat_id = (
                fold_index // 5 + 1
            )

            fold_id = (
                fold_index % 5 + 1
            )

            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_train = y[
                train_idx
            ]

            y_test = y[
                test_idx
            ]

            pipeline = (
                make_poly_svr_pipeline(
                    numeric_features
                )
            )

            # ==============================================
            # INNER GRID SEARCH
            # ==============================================

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,

                scoring=(
                    "neg_mean_absolute_error"
                ),

                cv=inner_cv,

                refit=True,

                n_jobs=-1,

                error_score="raise",

                verbose=0
            )

            # Hyperparameter tuning:
            # outer-training data only
            search.fit(
                X_train,
                y_train
            )

            # ==============================================
            # OUTER TEST PREDICTION
            # ==============================================

            y_pred = (
                search.predict(
                    X_test
                )
            )

            best_params = (
                search.best_params_
            )

            repetition_predictions[
                repeat_id
            ][test_idx] = y_pred

            # ==============================================
            # SAVE FOLD RESULTS
            # ==============================================

            fold_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "Fold":
                    fold_id,

                "Train_N":
                    len(train_idx),

                "Test_N":
                    len(test_idx),

                "Best_C":
                    best_params[
                        "svr__C"
                    ],

                "Best_Epsilon":
                    best_params[
                        "svr__epsilon"
                    ],

                "Best_Gamma":
                    best_params[
                        "svr__gamma"
                    ],

                "Best_Degree":
                    best_params[
                        "svr__degree"
                    ],

                "Best_Coef0":
                    best_params[
                        "svr__coef0"
                    ],

                "Test_MAE":
                    mean_absolute_error(
                        y_test,
                        y_pred
                    )
            })

            # ==============================================
            # SAVE INDIVIDUAL PREDICTIONS
            # ==============================================

            for (
                local_idx,
                actual,
                predicted
            ) in zip(
                test_idx,
                y_test,
                y_pred
            ):

                prediction_records.append({
                    "Target":
                        target,

                    "Feature_Set":
                        set_name,

                    "Repeat":
                        repeat_id,

                    "Fold":
                        fold_id,

                    "Excel_Row":
                        int(
                            valid_df.index[
                                local_idx
                            ]
                        ) + 2,

                    "Actual":
                        actual,

                    "Predicted":
                        predicted,

                    "Residual":
                        actual - predicted
                })

            # Progress output
            if fold_id == 5:

                elapsed_min = (
                    time.time()
                    - start_time
                ) / 60

                print(
                    f"  Repeat "
                    f"{repeat_id}/10 completed "
                    f"| Elapsed: "
                    f"{elapsed_min:.1f} min"
                )

        # ==================================================
        # 9. METRICS FOR EACH COMPLETE REPETITION
        # ==================================================

        for (
            repeat_id,
            predictions
        ) in repetition_predictions.items():

            if np.isnan(
                predictions
            ).any():

                raise RuntimeError(
                    "Incomplete OOF predictions"
                )

            mae = (
                mean_absolute_error(
                    y,
                    predictions
                )
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y,
                    predictions
                )
            )

            r2 = (
                r2_score(
                    y,
                    predictions
                )
            )

            repeat_records.append({
                "Target":
                    target,

                "Feature_Set":
                    set_name,

                "Repeat":
                    repeat_id,

                "N":
                    len(y),

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            })

        print(
            "Completed successfully."
        )

# ==================================================
# 10. CREATE RESULT DATAFRAMES
# ==================================================

repeat_df = pd.DataFrame(
    repeat_records
)

fold_df = pd.DataFrame(
    fold_records
)

prediction_df = pd.DataFrame(
    prediction_records
)

# ==================================================
# 11. FINAL PERFORMANCE SUMMARY
# ==================================================

summary = (
    repeat_df
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .agg(
        N=("N", "first"),

        MAE_Mean=(
            "MAE",
            "mean"
        ),

        MAE_SD=(
            "MAE",
            "std"
        ),

        RMSE_Mean=(
            "RMSE",
            "mean"
        ),

        RMSE_SD=(
            "RMSE",
            "std"
        ),

        R2_Mean=(
            "R2",
            "mean"
        ),

        R2_SD=(
            "R2",
            "std"
        )
    )
    .reset_index()
)

# ==================================================
# 12. HYPERPARAMETER FREQUENCY TABLES
# ==================================================

def selection_summary(
    df_in,
    parameter
):

    return (
        df_in
        .groupby([
            "Target",
            "Feature_Set",
            parameter
        ])
        .size()
        .reset_index(
            name="Selection_Count"
        )
        .sort_values(
            [
                "Target",
                "Feature_Set",
                "Selection_Count"
            ],
            ascending=[
                True,
                True,
                False
            ]
        )
    )

C_summary = selection_summary(
    fold_df,
    "Best_C"
)

epsilon_summary = selection_summary(
    fold_df,
    "Best_Epsilon"
)

gamma_summary = selection_summary(
    fold_df,
    "Best_Gamma"
)

degree_summary = selection_summary(
    fold_df,
    "Best_Degree"
)

coef0_summary = selection_summary(
    fold_df,
    "Best_Coef0"
)

# ==================================================
# 13. FULL PARAMETER COMBINATIONS
# ==================================================

parameter_summary = (
    fold_df
    .groupby([
        "Target",
        "Feature_Set",
        "Best_C",
        "Best_Epsilon",
        "Best_Gamma",
        "Best_Degree",
        "Best_Coef0"
    ])
    .size()
    .reset_index(
        name="Selection_Count"
    )
    .sort_values(
        [
            "Target",
            "Feature_Set",
            "Selection_Count"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

# ==================================================
# 14. DISPLAY RESULTS
# ==================================================

print("\n" + "=" * 90)

print(
    "TUNED POLY-SVR NESTED CV — FINAL SUMMARY"
)

print("=" * 90)

print(
    summary
    .round(3)
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("DEGREE SELECTION FREQUENCY")
print("=" * 90)

print(
    degree_summary
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("COEF0 SELECTION FREQUENCY")
print("=" * 90)

print(
    coef0_summary
    .to_string(index=False)
)

print("\n" + "=" * 100)

print(
    "TOP POLY-SVR HYPERPARAMETER COMBINATIONS"
)

print("=" * 100)

print(
    parameter_summary
    .groupby([
        "Target",
        "Feature_Set"
    ])
    .head(10)
    .to_string(index=False)
)

# ==================================================
# 15. SAVE RESULTS
# ==================================================

output_file = (
    "SVR_Tuned_Poly_"
    "NestedCV_Results.xlsx"
)

with pd.ExcelWriter(
    output_file
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    repeat_df.to_excel(
        writer,
        sheet_name="Repeat_Metrics",
        index=False
    )

    fold_df.to_excel(
        writer,
        sheet_name="Fold_Details",
        index=False
    )

    prediction_df.to_excel(
        writer,
        sheet_name="Predictions",
        index=False
    )

    C_summary.to_excel(
        writer,
        sheet_name="C_Selection",
        index=False
    )

    epsilon_summary.to_excel(
        writer,
        sheet_name="Epsilon_Selection",
        index=False
    )

    gamma_summary.to_excel(
        writer,
        sheet_name="Gamma_Selection",
        index=False
    )

    degree_summary.to_excel(
        writer,
        sheet_name="Degree_Selection",
        index=False
    )

    coef0_summary.to_excel(
        writer,
        sheet_name="Coef0_Selection",
        index=False
    )

    parameter_summary.to_excel(
        writer,
        sheet_name="Parameter_Selection",
        index=False
    )

# ==================================================
# 16. TOTAL RUNNING TIME
# ==================================================

total_minutes = (
    time.time()
    - start_time
) / 60

print(
    f"\nSaved: {output_file}"
)

print(
    "Tuned Polynomial-SVR Nested CV completed."
)

print(
    "Original dataset unchanged."
)

print(
    f"Total running time: "
    f"{total_minutes:.1f} minutes"
)


TARGET: Rp0.2_Mpa
VALID SAMPLES: 28

Running Tuned Poly-SVR: Compact
  Repeat 1/10 completed | Elapsed: 1.1 min
  Repeat 2/10 completed | Elapsed: 2.0 min
  Repeat 3/10 completed | Elapsed: 2.8 min
  Repeat 4/10 completed | Elapsed: 3.7 min
  Repeat 5/10 completed | Elapsed: 4.6 min
  Repeat 6/10 completed | Elapsed: 5.5 min
  Repeat 7/10 completed | Elapsed: 6.3 min
  Repeat 8/10 completed | Elapsed: 7.2 min
  Repeat 9/10 completed | Elapsed: 8.0 min
  Repeat 10/10 completed | Elapsed: 8.9 min
Completed successfully.

Running Tuned Poly-SVR: Full_Chemistry
  Repeat 1/10 completed | Elapsed: 9.7 min
  Repeat 2/10 completed | Elapsed: 10.6 min
  Repeat 3/10 completed | Elapsed: 11.4 min
  Repeat 4/10 completed | Elapsed: 12.3 min
  Repeat 5/10 completed | Elapsed: 13.2 min
  Repeat 6/10 completed | Elapsed: 14.0 min
  Repeat 7/10 completed | Elapsed: 14.8 min
  Repeat 8/10 completed | Elapsed: 15.7 min
  Repeat 9/10 completed | Elapsed: 16.6 min
  Repeat 10/10 completed | Elapsed: 17.5

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# 1. RESULT FILES
# ==========================================================

files = {
    "Dummy": "Dummy_Baseline_Results.xlsx",
    "Ridge": "Ridge_Tuned_Alpha_Solver_NestedCV_Results.xlsx",
    "PLS": "PLS_Tuned_FullGrid_NestedCV_Results.xlsx",
    "RBF_SVR": "SVR_Tuned_RBF_NestedCV_Results.xlsx",
    "Poly_SVR": "SVR_Tuned_Poly_NestedCV_Results.xlsx"
}

# Check that all files exist
for model_name, file_name in files.items():
    if not os.path.exists(file_name):
        raise FileNotFoundError(
            f"Missing file: {file_name}"
        )

print("All result files found.")

# ==========================================================
# 2. LOAD FINAL SUMMARY TABLES
# ==========================================================

summary_tables = []

for model_name, file_name in files.items():

    temp = pd.read_excel(
        file_name,
        sheet_name="Summary"
    )

    temp.insert(
        0,
        "Model",
        model_name
    )

    # Dummy has no Feature_Set
    if "Feature_Set" not in temp.columns:
        temp.insert(
            2,
            "Feature_Set",
            "Baseline"
        )

    summary_tables.append(temp)

final_summary = pd.concat(
    summary_tables,
    ignore_index=True
)

# Sort by target and MAE
final_summary = final_summary.sort_values(
    ["Target", "MAE_Mean"]
).reset_index(drop=True)

print("\n" + "=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

print(
    final_summary.round(3).to_string(index=False)
)

# ==========================================================
# 3. CALCULATE IMPROVEMENT RELATIVE TO DUMMY
# ==========================================================

dummy_mae = (
    final_summary[
        final_summary["Model"] == "Dummy"
    ]
    .set_index("Target")["MAE_Mean"]
    .to_dict()
)

final_summary["MAE_Reduction_vs_Dummy_pct"] = (
    final_summary.apply(
        lambda row:
        100 * (
            dummy_mae[row["Target"]]
            - row["MAE_Mean"]
        )
        / dummy_mae[row["Target"]],
        axis=1
    )
)

print("\n" + "=" * 100)
print("PERFORMANCE RELATIVE TO DUMMY BASELINE")
print("=" * 100)

print(
    final_summary[
        [
            "Target",
            "Model",
            "Feature_Set",
            "MAE_Mean",
            "RMSE_Mean",
            "R2_Mean",
            "MAE_Reduction_vs_Dummy_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

# ==========================================================
# 4. LOAD REPEAT-LEVEL RESULTS
# ==========================================================

ridge_repeat = pd.read_excel(
    files["Ridge"],
    sheet_name="Repeat_Metrics"
)

pls_repeat = pd.read_excel(
    files["PLS"],
    sheet_name="Repeat_Metrics"
)

rbf_repeat = pd.read_excel(
    files["RBF_SVR"],
    sheet_name="Repeat_Metrics"
)

poly_repeat = pd.read_excel(
    files["Poly_SVR"],
    sheet_name="Repeat_Metrics"
)

# ==========================================================
# 5. PAIRED REPEAT COMPARISON FUNCTION
# ==========================================================

def compare_repeats(
    df_a,
    model_a,
    df_b,
    model_b,
    target,
    feature_set
):

    a = df_a[
        (df_a["Target"] == target)
        &
        (df_a["Feature_Set"] == feature_set)
    ][
        ["Repeat", "MAE", "RMSE", "R2"]
    ].copy()

    b = df_b[
        (df_b["Target"] == target)
        &
        (df_b["Feature_Set"] == feature_set)
    ][
        ["Repeat", "MAE", "RMSE", "R2"]
    ].copy()

    merged = a.merge(
        b,
        on="Repeat",
        suffixes=(
            f"_{model_a}",
            f"_{model_b}"
        )
    )

    mae_a = f"MAE_{model_a}"
    mae_b = f"MAE_{model_b}"

    merged["MAE_Difference"] = (
        merged[mae_a]
        - merged[mae_b]
    )

    wins_a = (
        merged[mae_a]
        <
        merged[mae_b]
    ).sum()

    wins_b = (
        merged[mae_b]
        <
        merged[mae_a]
    ).sum()

    ties = len(merged) - wins_a - wins_b

    print("\n" + "=" * 90)
    print(
        f"{target} | {feature_set}: "
        f"{model_a} vs {model_b}"
    )
    print("=" * 90)

    print(
        f"{model_a} lower MAE in "
        f"{wins_a}/10 repeats"
    )

    print(
        f"{model_b} lower MAE in "
        f"{wins_b}/10 repeats"
    )

    print(
        f"Ties: {ties}/10"
    )

    print(
        f"Mean MAE difference "
        f"({model_a} - {model_b}): "
        f"{merged['MAE_Difference'].mean():.2f} MPa"
    )

    return merged

# ==========================================================
# 6. Rp0.2: PLS vs RBF-SVR
# ==========================================================

rp_pls_vs_rbf = compare_repeats(
    pls_repeat,
    "PLS",
    rbf_repeat,
    "RBF",
    target="Rp0.2_Mpa",
    feature_set="Compact"
)

# ==========================================================
# 7. Rm: PLS vs RIDGE
# ==========================================================

rm_pls_vs_ridge = compare_repeats(
    pls_repeat,
    "PLS",
    ridge_repeat,
    "Ridge",
    target="Rm_Mpa",
    feature_set="Full_Chemistry"
)

# ==========================================================
# 8. Rm: PLS vs RBF-SVR
# ==========================================================

rm_pls_vs_rbf = compare_repeats(
    pls_repeat,
    "PLS",
    rbf_repeat,
    "RBF",
    target="Rm_Mpa",
    feature_set="Full_Chemistry"
)

# ==========================================================
# 9. LOAD PREDICTION TABLES
# ==========================================================

pls_pred = pd.read_excel(
    files["PLS"],
    sheet_name="Predictions"
)

rbf_pred = pd.read_excel(
    files["RBF_SVR"],
    sheet_name="Predictions"
)

ridge_pred = pd.read_excel(
    files["Ridge"],
    sheet_name="Predictions"
)

# ==========================================================
# 10. LOAD ORIGINAL DATA FOR IDENTIFICATION
# ==========================================================

github_url = (
    "https://raw.githubusercontent.com/"
    "alimor37/WAAM_Weld_ML/main/"
    "Cleaned_data_numeric.xlsx"
)

df_original = pd.read_excel(
    github_url
)

df_original["Excel_Row"] = (
    df_original.index + 2
)

sample_info = df_original[
    [
        "Excel_Row",
        "Weld_ID",
        "Base_Material",
        "Weld_Method"
    ]
].copy()

# ==========================================================
# 11. AGGREGATE OOF PREDICTIONS ACROSS 10 REPEATS
# ==========================================================

def aggregate_predictions(
    prediction_df,
    target,
    feature_set,
    model_name
):

    temp = prediction_df[
        (prediction_df["Target"] == target)
        &
        (
            prediction_df["Feature_Set"]
            == feature_set
        )
    ].copy()

    temp["Absolute_Error"] = (
        temp["Residual"].abs()
    )

    result = (
        temp
        .groupby(
            [
                "Excel_Row",
                "Actual"
            ]
        )
        .agg(
            Predicted_Mean=(
                "Predicted",
                "mean"
            ),
            Predicted_SD=(
                "Predicted",
                "std"
            ),
            Residual_Mean=(
                "Residual",
                "mean"
            ),
            Mean_Absolute_Error=(
                "Absolute_Error",
                "mean"
            )
        )
        .reset_index()
    )

    result["Model"] = model_name

    result = result.merge(
        sample_info,
        on="Excel_Row",
        how="left"
    )

    return result

# Rp0.2 selected candidates
rp_pls_predictions = aggregate_predictions(
    pls_pred,
    target="Rp0.2_Mpa",
    feature_set="Compact",
    model_name="PLS_Compact"
)

rp_rbf_predictions = aggregate_predictions(
    rbf_pred,
    target="Rp0.2_Mpa",
    feature_set="Compact",
    model_name="RBF_SVR_Compact"
)

# Rm selected main model
rm_pls_predictions = aggregate_predictions(
    pls_pred,
    target="Rm_Mpa",
    feature_set="Full_Chemistry",
    model_name="PLS_Full_Chemistry"
)

# Comparator for Rm
rm_ridge_predictions = aggregate_predictions(
    ridge_pred,
    target="Rm_Mpa",
    feature_set="Full_Chemistry",
    model_name="Ridge_Full_Chemistry"
)

# ==========================================================
# 12. DISPLAY LARGEST ERRORS
# ==========================================================

def print_largest_errors(
    result_df,
    title,
    n=5
):

    largest = (
        result_df
        .sort_values(
            "Mean_Absolute_Error",
            ascending=False
        )
        .head(n)
    )

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    print(
        largest[
            [
                "Excel_Row",
                "Weld_ID",
                "Base_Material",
                "Weld_Method",
                "Actual",
                "Predicted_Mean",
                "Predicted_SD",
                "Residual_Mean",
                "Mean_Absolute_Error"
            ]
        ]
        .round(2)
        .to_string(index=False)
    )

    return largest

rp_pls_largest = print_largest_errors(
    rp_pls_predictions,
    "TOP 5 ERRORS — Rp0.2 — PLS COMPACT"
)

rp_rbf_largest = print_largest_errors(
    rp_rbf_predictions,
    "TOP 5 ERRORS — Rp0.2 — RBF-SVR COMPACT"
)

rm_pls_largest = print_largest_errors(
    rm_pls_predictions,
    "TOP 5 ERRORS — Rm — PLS FULL CHEMISTRY"
)

# ==========================================================
# 13. ACTUAL vs PREDICTED PLOT FUNCTION
# ==========================================================

def actual_vs_predicted_plot(
    result_df,
    title,
    file_name
):

    plt.figure(
        figsize=(7, 6)
    )

    plt.scatter(
        result_df["Actual"],
        result_df["Predicted_Mean"],
        s=65,
        edgecolor="black",
        alpha=0.8
    )

    min_value = min(
        result_df["Actual"].min(),
        result_df["Predicted_Mean"].min()
    )

    max_value = max(
        result_df["Actual"].max(),
        result_df["Predicted_Mean"].max()
    )

    margin = (
        max_value - min_value
    ) * 0.05

    line_min = (
        min_value - margin
    )

    line_max = (
        max_value + margin
    )

    plt.plot(
        [line_min, line_max],
        [line_min, line_max],
        "--",
        linewidth=1.5
    )

    plt.xlabel(
        "Measured strength (MPa)"
    )

    plt.ylabel(
        "Predicted strength (MPa)"
    )

    plt.title(title)

    plt.xlim(
        line_min,
        line_max
    )

    plt.ylim(
        line_min,
        line_max
    )

    plt.tight_layout()

    plt.savefig(
        file_name,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ==========================================================
# 14. CREATE ACTUAL vs PREDICTED FIGURES
# ==========================================================

actual_vs_predicted_plot(
    rp_pls_predictions,
    "Rp0.2 — PLS Compact",
    "Rp02_PLS_Actual_vs_Predicted.png"
)

actual_vs_predicted_plot(
    rp_rbf_predictions,
    "Rp0.2 — RBF-SVR Compact",
    "Rp02_RBF_Actual_vs_Predicted.png"
)

actual_vs_predicted_plot(
    rm_pls_predictions,
    "Rm — PLS Full Chemistry",
    "Rm_PLS_Actual_vs_Predicted.png"
)

# ==========================================================
# 15. RESIDUAL PLOT FUNCTION
# ==========================================================

def residual_plot(
    result_df,
    title,
    file_name
):

    plt.figure(
        figsize=(7, 6)
    )

    plt.scatter(
        result_df["Actual"],
        result_df["Residual_Mean"],
        s=65,
        edgecolor="black",
        alpha=0.8
    )

    plt.axhline(
        y=0,
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "Measured strength (MPa)"
    )

    plt.ylabel(
        "Residual: Measured - Predicted (MPa)"
    )

    plt.title(title)

    plt.tight_layout()

    plt.savefig(
        file_name,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ==========================================================
# 16. CREATE RESIDUAL FIGURES
# ==========================================================

residual_plot(
    rp_pls_predictions,
    "Residuals — Rp0.2 — PLS Compact",
    "Rp02_PLS_Residuals.png"
)

residual_plot(
    rp_rbf_predictions,
    "Residuals — Rp0.2 — RBF-SVR Compact",
    "Rp02_RBF_Residuals.png"
)

residual_plot(
    rm_pls_predictions,
    "Residuals — Rm — PLS Full Chemistry",
    "Rm_PLS_Residuals.png"
)

# ==========================================================
# 17. SAVE FINAL ANALYSIS
# ==========================================================

output_file = (
    "Final_Model_Comparison_and_Residuals.xlsx"
)

with pd.ExcelWriter(
    output_file
) as writer:

    final_summary.to_excel(
        writer,
        sheet_name="Final_Comparison",
        index=False
    )

    rp_pls_vs_rbf.to_excel(
        writer,
        sheet_name="Rp_PLS_vs_RBF",
        index=False
    )

    rm_pls_vs_ridge.to_excel(
        writer,
        sheet_name="Rm_PLS_vs_Ridge",
        index=False
    )

    rm_pls_vs_rbf.to_excel(
        writer,
        sheet_name="Rm_PLS_vs_RBF",
        index=False
    )

    rp_pls_predictions.to_excel(
        writer,
        sheet_name="Rp_PLS_Predictions",
        index=False
    )

    rp_rbf_predictions.to_excel(
        writer,
        sheet_name="Rp_RBF_Predictions",
        index=False
    )

    rm_pls_predictions.to_excel(
        writer,
        sheet_name="Rm_PLS_Predictions",
        index=False
    )

    rm_ridge_predictions.to_excel(
        writer,
        sheet_name="Rm_Ridge_Predictions",
        index=False
    )

print("\n" + "=" * 100)
print("FINAL ANALYSIS COMPLETED")
print("=" * 100)

print(
    f"Saved: {output_file}"
)

print(
    "No model was retrained."
)

print(
    "No original data were modified."
)

FileNotFoundError: Missing file: Dummy_Baseline_Results.xlsx